<a href="https://colab.research.google.com/github/ricardoyx12/tareas-IA/blob/main/Copia_de_Asignacion_1_KNN_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Descripción del dataset: Pima Indians Diabetes

El **Pima Indians Diabetes Dataset** es un conjunto de datos clásico en Machine Learning y bioestadística, recopilado por el *National Institute of Diabetes and Digestive and Kidney Diseases*.  
Su propósito es **predecir la aparición de diabetes tipo 2** en mujeres de origen **pima** (una población indígena del sur de Arizona, EE.UU.), a partir de diversas variables clínicas y demográficas.

### Características principales:
- **Número de registros:** 392 (en esta versión limpia, el original tenía 768).  
- **Número de atributos (features):** 8 variables predictoras + 1 variable objetivo.  
- **Población:** Mujeres de al menos 21 años de edad de la etnia Pima.  
- **Tarea principal:** Clasificación binaria → determinar si una paciente tiene diabetes (`Outcome = 1`) o no (`Outcome = 0`).

### Variables:
1. **Pregnancies** → Número de embarazos.  
2. **Glucose** → Concentración de glucosa en plasma después de 2 horas en una prueba de tolerancia a la glucosa.  
3. **BloodPressure** → Presión arterial diastólica (mm Hg).  
4. **SkinThickness** → Espesor del pliegue cutáneo del tríceps (mm).  
5. **Insulin** → Nivel sérico de insulina (mu U/ml).  
6. **BMI** → Índice de masa corporal (peso en kg / altura² en m²).  
7. **DiabetesPedigreeFunction** → Probabilidad de diabetes basada en antecedentes familiares.  
8. **Age** → Edad en años.  
9. **Outcome** → Variable objetivo:  
   - `0` = No tiene diabetes  
   - `1` = Tiene diabetes  

### Relevancia:
Este dataset es ampliamente utilizado en cursos de **Inteligencia Artificial y Machine Learning** para enseñar:
- Procesamiento y limpieza de datos biomédicos.  
- Métodos de clasificación supervisada (KNN, regresión logística, Random Forest, SVM, redes neuronales, etc.).  
- Importancia de la normalización y estandarización en algoritmos basados en distancias.  

---


## Paso 1: Cargar la base de datos  
Cargamos el CSV en un `DataFrame` de `pandas`. Si tu archivo no se llama exactamente `cleaned_dataset.csv`, ajusta la ruta.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score

# Ajusta la ruta si es necesario
df = pd.read_csv('Asignaciones/Asignacion_1/dataset/cleaned_dataset.csv')

# Breve descripción del dataset [cite: 1, 2]
print(f"Registros: {df.shape[0]} | Atributos: {df.shape[1]-1}")
df.head()

## Paso 2: Crear subconjuntos con 20 datos de **entrenamiento** y 20 de **testeo**
Seleccionaremos 40 muestras: 20 para entrenar y 20 para evaluar.

In [ ]:
# Variables predictoras (X) y objetivo (y) [cite: 10]
X_inicial = df.drop('Outcome', axis=1).values
y_inicial = df['Outcome'].values

X_train_small = X_inicial[:20]
X_test_small = X_inicial[20:40]
y_train_small = y_inicial[:20]
y_test_small = y_inicial[20:40]

## Paso 3: Implementar la función de distancia euclidiana

**Instrucciones:**
- Escribe una función en Python que reciba dos vectores y calcule la distancia euclidiana entre ellos.
- Utiliza la siguiente fórmula matemática para la distancia euclidiana entre dos vectores $x$ y $y$ de $n$ dimensiones:

$$
d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}
$$

- Prueba tu función con los siguientes dos ejemplos (cada vector corresponde a una fila del dataset):

| Embarazos | Glucosa | Presión Arterial | Grosor Piel | Insulina | IMC  | Función Hereditaria | Edad | Resultado |
|-----------|---------|------------------|-------------|----------|------|---------------------|------|-----------|
|     1     |   106   |        70        |      28     |   135    | 34.2 |        0.142        |  22  |     0     |
|     2     |   102   |        86        |      36     |   120    | 45.5 |        0.127        |  23  |     1     |

- Calcula la distancia euclidiana a mano y luego verifica que el resultado de tu función sea el mismo.
- La función debe imprimir el resultado del cálculo de la distancia euclidiana con los datos presentados.



In [ ]:
def distancia_euclidiana(x, y):
    return np.sqrt(np.sum((x - y)**2))

# Prueba con los datos de la tabla del Paso 3 [cite: 17, 18]
v1 = np.array([1, 106, 70, 28, 135, 34.2, 0.142, 22])
v2 = np.array([2, 102, 86, 36, 120, 45.5, 0.127, 23])

dist = distancia_euclidiana(v1, v2)
print(f"Distancia Euclidiana calculada: {dist:.4f}")

## Paso 4: Implementar un clasificador KNN básico

**Instrucciones:**
- Escribe una función que, dado un punto de prueba, calcule la distancia a todos los puntos de entrenamiento utilizando tu función de distancia euclidiana.
- Selecciona los **k = 3** vecinos más cercanos y predice la clase mayoritaria entre ellos.
- Aplica tu función a las 10 muestras de prueba obtenidas previamente, utilizando las 10 muestras de entrenamiento como referencia.
- El script debe imprimir una tabla comparando el valor real de `Resultado` de cada muestra de prueba con el valor predicho por tu algoritmo.
- Considere que las tablas se pueden codificar con un formato similar al que se muestra en el siguiente código:

In [ ]:
def knn_manual(train_data, train_labels, test_point, k=3):
    distancias = []
    for i in range(len(train_data)):
        d = distancia_euclidiana(train_data[i], test_point)
        distancias.append((d, train_labels[i]))

    # Ordenar por distancia y tomar k vecinos
    distancias.sort(key=lambda x: x[0])
    vecinos = distancias[:k]

    # Votación mayoritaria
    votos = [v[1] for v in vecinos]
    return max(set(votos), key=votos.count)

# Comparativa de las 10 muestras de prueba [cite: 26]
print(f"{'Real':<10} | {'Predicho':<10}")
print("-" * 25)
for i in range(10):
    pred = knn_manual(X_train_small, y_train_small, X_test_small[i])
    print(f"{y_test_small[i]:<10} | {pred:<10}")

## Paso 5: Usar toda la data con separación 80% entrenamiento / 20% testeo  

### Pasos:
1. Cargar todo el dataset.  
2. Separar variables (X) y etiquetas (y).  
3. Aplicar `train_test_split` con 80% para entrenamiento y 20% para testeo.  
4. Mantener la proporción de clases usando estratificación.  
5. Guardar los conjuntos de datos para usarlos en KNN.  

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

resultados = {}

## Paso 6: Entrenar un KNN con los datos sin escalar (crudos) y calcular accuracy  

### Pasos:
1. Definir el valor de **k = 3** y el metodo **Euclidiano**.  
2. Entrenar el modelo KNN con los datos crudos (sin normalizar/estandarizar).  
3. Predecir las clases del conjunto de test.  
4. Calcular el **accuracy** comparando predicciones con etiquetas reales.  
5. Guardar el resultado para la tabla comparativa.  


In [ ]:
Datos crudos (sin escalar) [cite: 32, 33]
knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
knn.fit(X_train, y_train)
resultados['Sin Escalar'] = accuracy_score(y_test, knn.predict(X_test))

## Paso 7: Normalizar (Min-Max scaling) y entrenar KNN, luego calcular accuracy  

### Pasos:
1. Aplicar **normalización Min-Max** a los datos de entrenamiento y test.  
2. Entrenar el modelo KNN con los datos normalizados.  
3. Predecir las clases del conjunto de test.  
4. Calcular el **accuracy** del modelo.  
5. Guardar el resultado para la tabla comparativa.  


In [ ]:
Normalización (Min-Max) [cite: 37, 38]
scaler_mm = MinMaxScaler()
X_train_norm = scaler_mm.fit_transform(X_train)
X_test_norm = scaler_mm.transform(X_test)
knn.fit(X_train_norm, y_train)
resultados['Normalizado'] = accuracy_score(y_test, knn.predict(X_test_norm))

## Paso 9: Estandarizar (Z-score) y entrenar KNN, luego calcular accuracy  

### Pasos:
1. Aplicar **estandarización Z-score** a los datos de entrenamiento y test.  
2. Entrenar el modelo KNN con los datos estandarizados.  
3. Predecir las clases del conjunto de test.  
4. Calcular el **accuracy** del modelo.  
5. Guardar el resultado para la tabla comparativa.  


In [ ]:
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std = scaler_std.transform(X_test)
knn.fit(X_train_std, y_train)
resultados['Estandarizado'] = accuracy_score(y_test, knn.predict(X_test_std))

## Paso 10/11: Tabla comparativa de accuracies  

### Pasos:
1. Reunir los resultados de accuracy de cada experimento:  
   - KNN sin escalar (80/20).  
   - KNN normalizado (80/20).  
   - KNN estandarizado (80/20).  
2. Crear una tabla con los resultados.  
3. Comparar el desempeño de cada método.  



In [ ]:

df_comparativa = pd.DataFrame(list(resultados.items()), columns=['Método', 'Accuracy'])
print(df_comparativa)

---
## Preguntas de reflexión y aplicación



1. ¿Por qué es importante normalizar o estandarizar los datos antes de usar KNN?  



KNN se basa en distancias. Si una variable tiene rangos mucho mayores que otra (ej. Glucosa vs. Edad), dominará el cálculo y sesgará el modelo

2. ¿Qué diferencias observaste en el accuracy entre los datos crudos, normalizados y estandarizados?  


RespiLos datos normalizados o estandarizados suelen dar un mejor accuracy porque todas las variables clínicas contribuyen equitativamentenda aqui

3. Si aumentamos el valor de **k** (número de vecinos), ¿cómo crees que cambiaría el rendimiento del modelo?  


RespondaIncrementar k ayuda a reducir el impacto del ruido, pero si es demasiado alto, el modelo puede volverse demasiado simple y perder precisión en los bordes de las clases aqui

4. ¿Qué ventaja tiene implementar KNN manualmente antes de usar scikit-learn?  


Permite entender la lógica interna del algoritmo (cómo se calculan los vecinos y los votos) antes de usarlo como una "caja negrauesta aqui"

5. ¿Qué limitaciones presenta KNN cuando se aplica a conjuntos de datos grandes o con muchas dimensiones?  

KNN es costoso computacionalmente en datasets grandes y sufre con muchas dimensiones, ya que las distancias entre puntos tienden a volverse similares (la "maldición de la dimensionalidad")

---

## Rúbrica de evaluación: Práctica KNN

| Criterio | Descripción | Puntaje Máximo |
|----------|-------------|----------------|
| **1. Carga y exploración del dataset** | Carga correcta del archivo CSV, explicación de las variables y verificación de datos. | 15 pts |
| **2. Implementación manual de KNN** | Código propio para calcular distancias euclidianas, selección de vecinos y votación mayoritaria. | 20 pts |
| **3. Predicción individual (ejemplo aleatorio)** | Explicación clara del proceso paso a paso para un ejemplo de test. | 10 pts |
| **4. Uso de scikit-learn (KNN)** | Entrenamiento y evaluación con `train_test_split`, comparación con el método manual. | 15 pts |
| **5. Normalización y estandarización** | Aplicación correcta de Min-Max y Z-score, con cálculo de accuracy en cada caso. | 20 pts |
| **6. Tabla comparativa de accuracies** | Presentación clara de los resultados y comparación entre métodos. | 10 pts |
| **7. Reflexión y preguntas finales** | Respuestas a las preguntas de análisis planteadas (profundidad y claridad). | 10 pts |

**Total: 100 pts**
